# Predicting Nepal earthquake damage on buildings using Ensemble methods

In this exercise, you will use `Ensemble` learning to work on earthquakes again. However, this time you will use a different dataset and predict the damage inflicted on buildings in case of a major Earthquakes in Nepal. 

## Scientific Background
Every year The National Earthquake Information Center (NEIC) records an average of 20,000 earthquakes all around to world. This number is fairly large, and implies an average of 50 earthquake occured  each day$^{[1]}$. Of course, the magnitude of most of these Earthquakes is fairly low.

## Data
In 2015 April 25, an intense earthquake occured in Central Nepal at local time of 11:56 a.m. The data used here was collected through surveys by [Kathmandu Living Labs](http://www.kathmandulivinglabs.org/) and the [Central Bureau of Statistics](https://cbs.gov.np/). This data is one of the largest post-disaster dataset ever collected, and it includes a wide range of information.

## Objective
Your goal is to predict the level of damage sustained by different buildings during the 2015 ***Gorkha earthquake*** in Nepal. Compare ensemble methods using validation accuracy. A 70% benchmark is a discussion target, not a guaranteed result. 



# Opening the data

Open `earthquake_nepal.csv`. Drop the arbitrary `building_id` identifier, separate `damage_grade`, and reserve a stratified 20% test set (`random_state=42`). Split the remaining development data again, reserving 20% for validation. Fit a `LabelEncoder` on the training response, then transform the validation and test labels with it.

Use **validation** scores throughout the comparisons below. Keep test data untouched until the final model is chosen. This random split evaluates buildings from the sampled regions; unseen-region prediction would require a geographic group split.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Importing the dataset
data = pd.read_csv('../../Excercise_data/L06/earthquake_nepal.csv')

# Ensuring Data Quality
* Check for missing data
* Check what data types are present in your dataset: is it all `numerical`, or do you have `categorical` data?
* Plot a `heatmap` to check for strong co-linearitu


# Data Preprocessing
Build an **unfitted** `ColumnTransformer` named `preproc`:
- One-hot encode categorical predictors with `handle_unknown='ignore'`. Treat the three `geo_level_*_id` columns as categories rather than numerical distances.
- Standard-scale numerical predictors.
- Keep `X_train`, `X_val` and `X_test` in their original, untransformed form.

For each model use `Pipeline([('preprocessor', clone(preproc)), ('classifier', model)])`, importing `clone` from `sklearn.base`. This matters particularly for stacking: each base pipeline must learn its own preprocessing inside each internal training fold.

Use `random_state=42` for stochastic models and `max_iter=3000` for logistic regression. All scores below are on `X_val, y_val`.

In [ ]:
# 

# Linear Model


Fit a logistic-regression pipeline. Does validation accuracy reach the 70% discussion benchmark?

In [ ]:
#

# KNN model

Using a K-nearest neighbour classifier, do the same as above, and save the score as `knn_score`. Is `knn_score` > 70%?

In [ ]:
#

# Linear SVC using the `SGDClassifier` class

Finally, use a `linear SVC` by calling an instance of `SGDClassifier` and setting the paramaters of `loss="hinge"` and `max_iter=3000`. Save the score of this classifier as `sgd_score`. Is `sgd_score` > 70%?

In [ ]:
#

# Decision Trees

Finally, use a `DecisionTreeClassifier` to see how an individual decision tree would perform. Save the score of this classifier as `tree_score`. Is `tree_score` > 70%?

In [ ]:
#

# RandomForestClassifier

Fit a Random Forest pipeline and save its validation accuracy as `rf_score`. For this classroom comparison use `n_estimators=50`, `max_depth=12`, `min_samples_leaf=5` and `random_state=42`.

These bounds keep repeated voting and stacking fits manageable with the many one-hot-encoded geographic categories. They are fixed demonstration settings, not tuned optima. Does the resulting validation score reach the 70% discussion benchmark?

In [ ]:
#

# Bagging a KNN classifier
Random Forest trains trees on bootstrap samples and also randomly selects candidate features at each split. It need not outperform every other model on every dataset.

Try bagging `KNeighborsClassifier` using `BaggingClassifier` with `n_estimators=5`, `max_samples=0.5`, `max_features=0.5` and `random_state=42`. Wrap the bagging model in a preprocessing pipeline. Save validation accuracy as `knn_bag_score`.

Five estimators keep this demonstration manageable; trying 15 is optional. KNN prediction can be slow even though fitting is quick. Bagging is a parallel aggregation method, not gradient boosting.

In [ ]:
#

<details><summary>💡 Observations</summary>
Compare the actual validation scores. Did bagging help KNN? Explain why the number of members alone cannot determine whether an ensemble outperforms another model: member quality and error dependence also matter.
</details>

# Prediction using heterogenous classifiers

So far, we have used two `Ensemble` classifier: a `RandomForest` and a `Bagged KNN` classifier. Compare their validation performance with their single-model counterparts. But what they have in common is that they are composed of a homogeneous ensemble of classifiers (either only `DecisionTrees` or only `KNeighborsClassifier`).

Sometimes, it might be beneficial to use a heterogenous ensemble of classifiers to boost our performance. We can do this via two main classes in `sklearn`: the `VotingClassifier` and the `StackingClassifier`.

## `VotingClassifier`

Let's start with `VotingClassifier` as this is the most straightforward class. All it does is train a list of classifiers, and then agglomerate their results (either the predicted class (**hard voting**) or the predicted probabilities (**soft voting**)).<br>
Create a `VotingClassifier` composed of:
1. One `LogisticRegression` 
3. One `SGDClassifier(loss='hinge', max_iter=3000)`
4. One `KNeighborsClassifier`
5. One `RandomForestClassifier`

Use `voting='hard'` because the hinge-loss SGD classifier has no `predict_proba`. Use the four complete preprocessing-and-model pipelines as its estimators. Call this classifier `voting_classifier`, and save its performance in a variable called `voting_accuracy`. Do we outperform our `RandomForest` and the `bagged KNNs`?

In [ ]:
#

<details><summary>💡 Observations</summary>
Did equal-weight hard voting improve validation accuracy? If some members are less accurate or make the same errors, a vote can be worse than the best member. Weighting is a modelling choice to validate, not a guaranteed fix.
</details>

## `StackingClassifier`
Stacking fits a final model to predictions from base models. Use the same four **complete pipelines** as above and `LogisticRegression(max_iter=3000)` as `final_estimator`.

Use `cv=3` for the classroom run. The meta-model is trained on out-of-fold predictions, while each base pipeline refits its preprocessing within the internal training folds. Do not fit one preprocessor on all training observations before this step.

Fit on raw `X_train, y_train` and save validation accuracy as `stacking_accuracy`. Three folds limit runtime; five folds are an optional extension.

In [ ]:
#

<details><summary>💡 Observations</summary>
Compare stacking with the strongest individual pipeline. If it improves, discuss which complementary predictions may have helped. If it does not, what might limit the meta-model?
</details>

# XGBoost
Let's get acquainted with the basic use of the `XGBClassifier` class. Import it from the `xgboost` library, and run train a plain-vanilla version of the classifier. Use the same pipeline convention and `random_state=42`. Save validation accuracy as `xgb_accuracy`

In [ ]:
#

<details><summary>🔍 Observations</summary>
Compare the validation score with the previous models; no ranking is guaranteed.
</details>

# A validation set for early stopping
The validation set was reserved **before** preprocessing at the start of this notebook. Reuse that split rather than making another split of already transformed data.

XGBoost's `eval_set` does not automatically pass through a sklearn pipeline's preprocessing. Fit a fresh `clone(preproc)` on raw `X_train` only, then transform `X_train` and `X_val` into `X_train_prep` and `X_val_prep` using it. Retain this fitted transformer as `xgb_preproc`.

In [ ]:
# ADD YOUR CODE HERE

### `XGBoost` hyperparameters
- `n_estimators` is the maximum number of boosting rounds.
- `learning_rate` shrinks each new tree's contribution; it is not a neural-network weight update.
- `early_stopping_rounds` stops after a chosen number of rounds without validation improvement.

Create `xgb = XGBClassifier(n_estimators=100, learning_rate=1.0, eval_metric='mlogloss', early_stopping_rounds=10, random_state=42, n_jobs=2)`. The learning rate 1.0 is deliberately aggressive and lies within XGBoost's documented [0, 1] range.

Pass a list of `(X, y)` tuples to `fit(eval_set=...)`, with validation last. Use transformed features and unchanged labels; never pass test data for early stopping.

In [ ]:
# 

In [ ]:
# Complete the preceding model and preprocessing cells first.
if all(name in globals() for name in ['xgb', 'X_train_prep', 'X_val_prep', 'y_train', 'y_val']):
    xgb.fit(X_train_prep, y_train,
            eval_set=[(X_train_prep, y_train), (X_val_prep, y_val)])
else:
    print('Complete the XGBoost setup above to display the learning curve.')

## ☝️ Training performance at each iteration: learning curve

As you can see, `XGBoost` is outputing a log of the `losses` both for your training (`validation_0` if you passed `(X_train, y_train)` first in your `eval_set`) and your validation set (`validation_1` when the validation tuple is second).

If you remember last week's lecture on training curves (and I hope you do!) you will recognise the value of this data: we can plot a learning curve, and see what our algorthm is doing.

**Do the following:**
* See what the `evals_result()` function returns when applied to your trained `XGBClassifier` (*tip*: it returns the training loop losses for both of your validation sets). Save this into a variable called `results`
* Explore `results`: it is a dictionary-like objects, so see what the keys are, and what the keys of the returned object are.
* Once you understand your `results`, write a simple python function that will draw both training curves on a plot. I suggest using a `figsize` of 15, 10, or something similar to show the curve nicely.
* Draw the learning curve for the `XGBClassifier` that you trained above

Then, based on your observations, answer the following question:
* Does the algorithm `'underfit'`, `'overfit'`, or is it `'balanced'`?

Save your answer as a string in a variable named `performance`

In [ ]:
#

## Changing Hyperparameters

Now, let's change the hyperparameters of the `XGBClassifier` to:
* `eval_metric='mlogloss'` 
* `n_estimators=150` 
* `learning_rate=0.3`

Retrain your model, and draw the learning curve. What do you think your model is doing now?

In [ ]:
#

## Controlling overfitting
Compare the learning curves after changing `learning_rate` to 0.3 and `n_estimators` to 150. Then try `min_child_weight=6` and `max_depth=7`, keeping the validation split fixed.

How do the train/validation gap and best validation loss change? Do not assume these choices must improve performance. Select your final settings using validation results.

If selecting XGBoost, retain `best_iteration + 1` as the number of rounds, refit preprocessing on all development data, then refit a fresh XGBoost model for that fixed number of rounds **without** early stopping. Evaluate once on test data and save accuracy as `final_score`. If another model wins, refit its complete pipeline on development data instead.

In [ ]:
#

# How were these hyperparameters chosen?
The settings above are candidates to investigate, not universal recommendations. A search such as `BayesSearchCV` can compare further settings using training-data cross-validation. Keep preprocessing inside each fold and do not return to test data to tune the search.

Summarise which model you selected, the evidence used to choose it, and its final test accuracy. Which changes would you make if prediction had to transfer to a previously unseen region?